# TREAD-ONE NB03 — classifier diagnosis and image explorer

This is the next **TREAD-ONE** step. It uses the six existing selected/final
checkpoints and performs **no training, optimizer update, new test evaluation or
checkpoint selection**. It measures nominal train/validation predictions,
defined gate/background/occlusion interventions and bounded fixed-batch gradients.

## Kaggle settings and order

1. Attach `shanmuk4622/tire-dataset-prepared-phase2` directly from Kaggle.
2. Internet **ON**, accelerator **T4 ×2**; **GPU 0 only** is used sequentially for
   consistent diagnostic inference. GPU 1 is unused; this is not distributed training.
3. Enable Kaggle Secret **HF_TOKEN** with write access to `Shanmuk4622/tyre-wear-study`.
4. Run All. Existing Kaggle packages are retained; no shared dependency downgrade.
5. Keep at least **6 GiB free** in `/kaggle/working`. Approximately 390 MiB of
   old source evidence is read; all new outputs are direct CSV/JSON/native NPY/HTML.

The report includes probabilities, per-tyre/domain failures, mask/gate/evidence
overlays and interactive opacity/class controls. More logged records are not
additional independent tyres. User-assisted new-data collection is a later step.

## Persistence and stop behavior

Local complete inference units contain at most 32 images, with all five variants.
HF snapshots publish after each major checkpoint job, every 30 minutes and on
catchable Stop/error. Direct CSV plus JSON cursors reconstruct progress in a fresh
compatible session; only the last incomplete local unit is replayed.

Use this notebook unchanged to resume. Recipe, source and environment identities
are frozen into a diagnostic namespace; different environments do not silently
merge results. Only one notebook may own that namespace. A hard-killed writer's
lease expires after 45 minutes; catchable clean stop releases it immediately.
Hard kernel/container death cannot publish unsent work. Server backoff still applies.


In [1]:
import base64, gzip, json, os, signal, subprocess, sys, time
from pathlib import Path
from kaggle_secrets import UserSecretsClient
WORK=Path('/kaggle/working/treadone_diagnostic')
WORK.mkdir(parents=True,exist_ok=True)
os.environ['HF_TOKEN']=UserSecretsClient().get_secret('HF_TOKEN')
payload=json.loads(gzip.decompress(base64.b64decode('...')))
for name,body in payload.items():
    target=WORK/name
    assert '..' not in Path(name).parts and not Path(name).is_absolute()
    target.parent.mkdir(parents=True,exist_ok=True)
    target.write_bytes(base64.b64decode(body))
print('TREAD-ONE diagnostic sources ready; HF_TOKEN loaded without displaying it.')

def launch(name,*arguments):
    process=subprocess.Popen([sys.executable,'-B','-u',str(WORK/name),*arguments],start_new_session=True)
    try:
        code=process.wait()
    except KeyboardInterrupt:
        print('Stop requested. Asking the diagnostic to persist and publish completed units; keep this session open.')
        os.killpg(process.pid,signal.SIGINT)
        for attempt in range(36):
            try:
                process.wait(timeout=5)
                break
            except subprocess.TimeoutExpired:
                if attempt%4==0: print('Waiting for the final snapshot/server backoff...')
        if process.poll() is None:
            print('The child is still flushing. Do not start a duplicate notebook; forced kernel death can lose unpublished work.')
        raise
    if code:
        raise RuntimeError(f'{name} exited with status {code}; completed local/remote units remain resumable. Read the diagnostic traceback above.')


TREAD-ONE diagnostic sources ready; HF_TOKEN loaded without displaying it.


## 1. Verify/retrieve the pinned r1 source evidence
Legacy ZIPs are downloaded only as read-only historical input. No new ZIP is uploaded. Cached complete downloads are reused.

In [2]:
launch('analysis/fetch_completed_results.py','--bundles')

Fetched INDEX.json 1043 bytes
Fetched LEASE.json 26 bytes
Fetched CONTRACT.json 3765 bytes
Fetched OVERLAY.json 806991 bytes
Fetched ENVIRONMENT.json 466 bytes
Fetched DATA_PREFLIGHT.json 1785 bytes
Fetched report/REPORT.json 790504 bytes
Fetched report/EXPORT.json 4279 bytes
Fetched report/EVALUATION_seed1.json 262007 bytes
Fetched report/EVALUATION_seed2.json 262684 bytes
Fetched report/EVALUATION_seed3.json 263756 bytes
runs/seed1/latest.zip 32 MiB downloaded
runs/seed1/latest.zip 64 MiB downloaded
runs/seed1/latest.zip 96 MiB downloaded
runs/seed1/latest.zip 128 MiB downloaded
SHA256 verified bundle runs/seed1/latest.zip
runs/seed2/latest.zip 32 MiB downloaded
runs/seed2/latest.zip 64 MiB downloaded
runs/seed2/latest.zip 96 MiB downloaded
runs/seed2/latest.zip 128 MiB downloaded
SHA256 verified bundle runs/seed2/latest.zip
runs/seed3/latest.zip 32 MiB downloaded
runs/seed3/latest.zip 64 MiB downloaded
runs/seed3/latest.zip 96 MiB downloaded
SHA256 verified bundle runs/seed3/latest.

## 2. Run or resume the fixed-checkpoint diagnosis
Full train/validation cohorts, three seeds, both saved endpoints, five variants. It restores compatible remote progress before deciding what to skip.

In [3]:
launch('diagnostic/run.py','--publish')

TREAD-ONE diagnostic: 975c08501981976bb992620aba89478dda90d18505f05d3bbe43376baff3fa0d images/checkpoint: 467 jobs: 6 device: cuda:0 publishing: True
seed1-selected images 32 / 467 complete; direct local records retained.
seed1-selected images 64 / 467 complete; direct local records retained.
seed1-selected images 96 / 467 complete; direct local records retained.
seed1-selected images 128 / 467 complete; direct local records retained.
seed1-selected images 160 / 467 complete; direct local records retained.
seed1-selected images 192 / 467 complete; direct local records retained.
seed1-selected images 224 / 467 complete; direct local records retained.
seed1-selected images 256 / 467 complete; direct local records retained.
seed1-selected images 288 / 467 complete; direct local records retained.
seed1-selected images 320 / 467 complete; direct local records retained.
seed1-selected images 352 / 467 complete; direct local records retained.
seed1-selected images 384 / 467 complete; direct l

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  ...f044eb/gallery/arrays.npy:   4%|▍         | 1.41MB / 36.4MB            

Processing Files (0 / 1)      :   4%|▍         | 1.41MB / 36.4MB,  140kB/s  
New Data Upload               :   4%|▍         | 1.41MB / 36.4MB,  140kB/s  

Processing Files (0 / 1)      :  37%|███▋      | 13.4MB / 36.4MB, 1.32MB/s  
New Data Upload               :  37%|███▋      | 13.4MB / 36.4MB, 1.32MB/s  

Processing Files (0 / 1)      :  87%|████████▋ | 31.7MB / 36.4MB, 3.11MB/s  
New Data Upload               :  87%|████████▋ | 31.7MB / 36.4MB, 3.11MB/s  

Processing Files (0 / 1)      : 100%|█████████▉| 36.4MB / 36.4MB, 3.54MB/s  
New Data Upload               : 100%|█████████▉| 36.4MB / 36.4MB, 3.54MB/s  

  ...f044eb/gallery/arrays.npy: 100%|█████████▉| 36.4MB / 36.4MB            

  ...f044eb/gallery/arrays.npy: 100%|█████████▉| 36.4MB / 36.4MB            


Verified HF diagnostic snapshot: 6e0148d07c1a0692c1ab216905f84bd129ec9d15 files: 26
seed1-fixed_final images 32 / 467 complete; direct local records retained.
seed1-fixed_final images 64 / 467 complete; direct local records retained.
seed1-fixed_final images 96 / 467 complete; direct local records retained.
seed1-fixed_final images 128 / 467 complete; direct local records retained.
seed1-fixed_final images 160 / 467 complete; direct local records retained.
seed1-fixed_final images 192 / 467 complete; direct local records retained.
seed1-fixed_final images 224 / 467 complete; direct local records retained.
seed1-fixed_final images 256 / 467 complete; direct local records retained.
seed1-fixed_final images 288 / 467 complete; direct local records retained.
seed1-fixed_final images 320 / 467 complete; direct local records retained.
seed1-fixed_final images 352 / 467 complete; direct local records retained.
seed1-fixed_final images 384 / 467 complete; direct local records retained.
seed1-f

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  ...885d2e/gallery/arrays.npy:   5%|▍         | 3.47MB / 72.8MB            

Processing Files (0 / 3)      :   3%|▎         | 3.47MB /  103MB,   ???B/s  

Processing Files (0 / 3)      :  19%|█▉        | 19.4MB /  103MB, 1.92MB/s  

Processing Files (0 / 3)      :  34%|███▍      | 35.4MB /  103MB, 3.49MB/s  

Processing Files (0 / 3)      :  35%|███▌      | 36.4MB /  103MB, 3.54MB/s  


  ...-aa885d2e/csv/latents.csv:   1%|▏         |  269kB / 19.7MB            



  ...lassifier_Diagnostic.html:   1%|▏         |  148kB / 10.9MB            

  ...885d2e/gallery/arrays.npy:  51%|█████     | 36.9MB / 72.8MB            


  ...-aa885d2e/csv/latents.csv:   1%|▏         |  269kB / 19.7MB            



Processing Files (0 / 3)      :  36%|███▌      | 37.3MB /  103MB, 3.60MB/s  
New Data Upload               :   1%|▏         |  915kB / 67.0MB, 89

Verified HF diagnostic snapshot: b4239d3943e1142dce8ede948af41334eac299b4 files: 27
seed2-selected images 32 / 467 complete; direct local records retained.
seed2-selected images 64 / 467 complete; direct local records retained.
seed2-selected images 96 / 467 complete; direct local records retained.
seed2-selected images 128 / 467 complete; direct local records retained.
seed2-selected images 160 / 467 complete; direct local records retained.
seed2-selected images 192 / 467 complete; direct local records retained.
seed2-selected images 224 / 467 complete; direct local records retained.
seed2-selected images 256 / 467 complete; direct local records retained.
seed3-fixed_final images 448 / 467 complete; direct local records retained.
seed3-fixed_final images 467 / 467 complete; direct local records retained.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  ...e-8b455114/csv/points.csv:  26%|██▌       | 7.96MB / 30.6MB            


  ...-8b455114/csv/latents.csv:  13%|█▎        | 7.91MB / 59.3MB            



  ...8b455114/csv/evidence.csv:  43%|████▎     | 7.94MB / 18.3MB            

  ...e-8b455114/csv/points.csv:  26%|██▌       | 7.96MB / 30.6MB            


  ...-8b455114/csv/latents.csv:  13%|█▎        | 7.91MB / 59.3MB            



Processing Files (0 / 5)      :   7%|▋         | 23.8MB /  359MB,   ???B/s  




  ...455114/gallery/arrays.npy:   7%|▋         | 15.8MB /  219MB            





  ...lassifier_Diagnostic.html:  44%|████▍     | 14.4MB / 32.5MB            

  ...e-8b455114/csv/points.csv:  52%|█████▏    | 16.0MB / 30.6MB            


  ...-8b455114/csv/latents.csv:  27%|██▋       | 15.9MB / 59.3MB            



  ...8b455114/csv/evidence.csv:  70%|██████▉   | 12.7MB /

Verified HF diagnostic snapshot: 75400f019ba0c6741b5060925b9c956c0a1ca0a8 files: 27
DIAGNOSTIC COMPLETE: /kaggle/working/treadone_diagnostic/diagnostics/classifier_r1/975c08501981976bb992620aba89478dda90d18505f05d3bbe43376baff3fa0d/snapshots/0096-complete-8b455114


## 3. Inspect and retain the visual report
Download/open the self-contained HTML for the image explorer. It needs no external service. If Kaggle blocks interactive inline output, the downloaded HTML remains interactive in your browser.

In [4]:
import html
import pandas as pd
from IPython.display import HTML, FileLink, display
completed=sorted((WORK/'diagnostics/classifier_r1').glob('*/COMPLETE.json'),key=lambda p:p.stat().st_mtime)
assert completed, 'Diagnosis is not complete yet; rerun the diagnostic cell to resume.'
completion=json.loads(completed[-1].read_text())
snapshot=Path(completion['snapshot'])
report=snapshot/'TREAD_ONE_Classifier_Diagnostic.html'
display(pd.read_csv(snapshot/'summary/cohort_group_metrics.csv').query("level == 'cohort' and variant == 'normal'"))
display(FileLink(str(report)))
SHOW_INLINE_PREVIEW=False  # Full explorer is about 31 MiB; keep ordinary notebook outputs compact.
if SHOW_INLINE_PREVIEW:
    display(HTML('<iframe title="TREAD-ONE diagnostic explorer" sandbox="allow-scripts" style="width:100%;height:1100px;border:0" srcdoc="'+html.escape(report.read_text(),quote=True)+'"></iframe>'))
print('Download/open the HTML for interactive images. Set SHOW_INLINE_PREVIEW=True only if the notebook renderer supports the large inline report.')
print((completed[-1].parent/'PUBLICATION.json').read_text())
print('Preserved output folder:',snapshot)


,job,split,variant,level,group,images,accuracy,balanced_accuracy_observed,nll,brier,mean_two_region_dice,human_point_images,human_point_mae
14,seed1-fixed_final,train,normal,cohort,all,386,1.000000,1.000000,0.082494,0.012576,0.990864,70,0.004195
74,seed1-fixed_final,validation,normal,cohort,all,81,0.234568,0.231707,2.020776,1.230046,0.989801,25,0.009738
104,seed1-selected,train,normal,cohort,all,386,1.000000,1.000000,0.293761,0.122247,0.988241,70,0.005963
164,seed1-selected,validation,normal,cohort,all,81,0.555556,0.550000,0.735170,0.427161,0.989970,25,0.011161
194,seed2-fixed_final,train,normal,cohort,all,386,1.000000,1.000000,0.057592,0.004852,0.991101,70,0.004132
254,seed2-fixed_final,validation,normal,cohort,all,81,0.246914,0.243902,2.074044,1.173668,0.989781,25,0.018314
284,seed2-selected,train,normal,cohort,all,386,1.000000,1.000000,0.113833,0.018473,0.989086,70,0.005713
344,seed2-selected,validation,normal,cohort,all,81,0.827160,0.825000,0.537220,0.280098,0.989477,25,0.019376
374,seed3-fixed_final,train,normal,cohort,all,386,1.000000,1.000000,0.140789,0.030383,0.990447,70,0.004579
434,seed3-fixed_final,validation,normal,cohort,all,81,0.024691,0.024390,1.949844,1.219343,0.989584,25,0.012314


/kaggle/working/treadone_diagnostic/diagnostics/classifier_r1/975c08501981976bb992620aba89478dda90d18505f05d3bbe43376baff3fa0d/snapshots/0096-complete-8b455114/TREAD_ONE_Classifier_Diagnostic.html

Download/open the HTML for interactive images. Set SHOW_INLINE_PREVIEW=True only if the notebook renderer supports the large inline report.
{"destination":"phase3/tread-one-r1/371c40c0c6aaedaf5c5970f7721a284b042494db7bb330065f6cbde8fc3a0b8c/diagnostics/classifier-r1/975c08501981976bb992620aba89478dda90d18505f05d3bbe43376baff3fa0d","files":27,"owner_released":true,"revision":"75400f019ba0c6741b5060925b9c956c0a1ca0a8","status":"published_and_verified","zip_uploads":0}
Preserved output folder: /kaggle/working/treadone_diagnostic/diagnostics/classifier_r1/975c08501981976bb992620aba89478dda90d18505f05d3bbe43376baff3fa0d/snapshots/0096-complete-8b455114
